## Grimm's Fairy Tales
I was looking at the Grimms' Fairy Tale and want to play with it.
link: https://www.gutenberg.org/cache/epub/2591/pg2591-images.html

Then I remember old the old ladies in the stories are usually bad or evil or dead(mothers) so I want to see of I can use Markov chain to generate a new stories for the old ladies

In [2]:
%pip install sentencex

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
!{sys.executable} -m pip install https://github.com/aparrish/shoestrings/archive/main.zip

  Using cached https://github.com/aparrish/shoestrings/archive/main.zip (26 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import sentencex
import re
import random
import textwrap


from shoestrings import TextGenerator
from shoestrings.tokenizers import SimpleWordTokenizer, SimpleCharacterTokenizer

text_fairy = open("grimm.txt").read()
sentences = sentencex.segment("en", text_fairy) 
sentences = [item.replace( "\n", " ").replace("_", "").replace('"', "") for item in sentences if len(item.replace( "\n", " ").replace("_", "").replace('"', "").strip()) > 0]

#cleaning up random punctuation
sentences = [re.sub(r"[^\w\s,'\-]", '', item) for item in sentences]

sentences

#see what old lady are up to
lady_sentences = [s for s in sentences if any(w in s.lower() for w in 
    ['old woman', 'witch', 'stepmother', 'Enchantress'])]

lady_sentences

['All the day long she flew about in the form of an owl, or crept about the country like a cat but at night she always became an old woman again ',
 'At this the robber ran back as fast as he could to his comrades, and told the captain how a horrid witch had got into the house, and had spat at him and scratched his face with her long bony fingers how a man with a knife in his hand had hidden himself behind the door, and stabbed him in the leg how a black monster stood in the yard and struck him with a club, and how the devil had sat upon the top of the house and cried out, Throw the rascal up here ',
 ' In a village dwelt a poor old woman, who had gathered together a dish of beans and wanted to cook them ',
 'The bean said I too have escaped with a whole skin, but if the old woman had got me into the pan, I should have been made into broth without any mercy, like my comrades ',
 'The old woman has destroyed all my brethren in fire and smoke she seized sixty of them at once, and took th


## 🧚‍♀️🧚‍♀️🧚‍♀️ Previous work : Assignment 3 - fairy tale 🧚‍♀️🧚‍♀️🧚‍♀️



In [5]:
#adding opening
#focusing on lady's story
def my_boost_he(token, sequence, score):
    if token.startswith('he') or token.startswith('He')  :
        return score - 100
    elif token.startswith('we') or token.startswith('We'):
        return score - 100
    elif token.startswith('I'):
        return score - 100
    elif token.startswith('me'):
        return score - 100
    elif token.startswith('my') or token.startswith('My'):
        return score - 100
    elif token.startswith('his') or token.startswith('His'):
        return score - 100
    elif token.startswith('him'):
        return score - 100
    else:
        return score

full_story = []

for attempt in range(100):
    lady_gen = TextGenerator(n=3, tokenizer=SimpleWordTokenizer(), source=lady_sentences)

    open_line = lady_gen.generate_one(start_string="a witch", temperature = 2,custom_scorer=my_boost_he)
    open_line = "Once upon a time there was " + open_line

    open_line = open_line + "."
    open_line = open_line.replace(" .", ".")
    open_line = open_line.lower().capitalize()
    
    if 10<len(open_line.split()) <= 20 :
        break

full_story.append(open_line)


#lady_gen = TextGenerator(n=6, tokenizer=SimpleWordTokenizer(), source=lady_sentences)
gen = TextGenerator(n=3, tokenizer=SimpleWordTokenizer(), source=sentences)

story = gen.generate_one(start_string= "she was", custom_scorer=my_boost_he, temperature=1.5, max_tokens = 10)
story = story + "."
story = story.replace(" .", ".")
story = story.lower().capitalize()
full_story.append(story)
    
story = gen.generate_one(start_string= "she thought", custom_scorer=my_boost_he)
story = story + "."
story = story.replace(" .", ".")
story = story.lower().capitalize()
full_story.append(story)

    
story = gen.generate_one(start_string= "the witch", custom_scorer=my_boost_he, beam_width=15, temperature=1.5)
story = story + "."
story = story.replace(" .", ".")
story = story.lower().capitalize()
full_story.append(story)


story = gen.generate_one(start_string= "she said", custom_scorer=my_boost_he)
story = story + "."
story = story.replace(" .", ".")
story = story.lower().capitalize()
full_story.append(story)

story = gen.generate_one( start_string= "then she",custom_scorer=my_boost_he, beam_width=15, temperature=1.5)
story = story + ", happily."
story = story.replace(" .", ".")
story = story.lower().capitalize()
full_story.append(story)
    
for line in full_story:
    line = line.replace("witch", "🧚‍♀️ witch 🧚‍♀️")
    print(textwrap.fill(line, width=100))
    print()



Once upon a time there was a 🧚‍♀️ witch 🧚‍♀️, who are you and what is your business.

She was the youngest kid ran with the same luck.

She thought it was impossible to awaken him.

The 🧚‍♀️ witch 🧚‍♀️ cried where are your commands.

She said she, how can you do not give up the mountain, and thrust into his kitchen and was very
angry, and the thieves come along, and then she said tomorrow morning before they had disappeared,
so deep that she fell on the porch.

Then she grasped the axe, split the anvil with one blow struck an anvil into the forest, happily.



In [6]:
%pip install simpleneighbors scikit-learn numpy


Note: you may need to restart the kernel to use updated packages.


## Processing the Dataset

In [15]:
lines = open("./semantic-categories-2026-04-10.jsonl").readlines()  # read each line into a list of strings
from collections import defaultdict, Counter
import json

# create the dictionary (unknown keys default to an empty list)
cats = defaultdict(list)

for item in lines: 
    
    obj = json.loads(item)  # decode the JSON
    
    if len(obj['label']) > 0:
        # optional: merge variations in case (e.g. DAY and day will be counted together)
        lc_text = obj['text'].lower()
        # the extend function appends the contents of one list to another
        cats[lc_text].extend(obj['label'])
list(cats.items())[:10]

[('excuse',
  ['cold', 'emotive', 'organic', 'sharp', 'tangible', 'academic', 'youthful']),
 ('kindred', ['academic', 'emotive', 'warm', 'warm', 'emotive']),
 ('file', ['academic', 'cold', 'tangible', 'academic', 'tangible']),
 ('surface', ['academic', 'round', 'tangible', 'tangible']),
 ('grease',
  ['outlandish', 'tangible', 'warm', 'round', 'warm', 'tangible', 'warm']),
 ('laughter',
  ['organic',
   'warm',
   'youthful',
   'organic',
   'round',
   'tangible',
   'warm',
   'youthful']),
 ('passion', ['emotive', 'emotive']),
 ('day',
  ['organic',
   'round',
   'warm',
   'warm',
   'youthful',
   'organic',
   'tangible',
   'warm',
   'organic',
   'youthful']),
 ('genius',
  ['academic',
   'cold',
   'sharp',
   'academic',
   'academic',
   'cold',
   'academic',
   'organic',
   'academic',
   'youthful']),
 ('home', ['emotive', 'warm', 'emotive', 'warm'])]

In [8]:
cats_counted = {}
for key, val in cats.items():
    cats_counted[key] = Counter(val)
## finding nice warm word in the tagged dataset
positive_words = [key for key, val in cats_counted.items() if val['warm'] > 0 or val['youthful'] > 0 or val['organic'] > 0]
print(positive_words)
## finding negative word in the tagged dataset
negative_words= [key for key, val in cats_counted.items() if val['sharp'] > 0 or val['cold'] > 0 or val['outlandish'] > 0]
print(negative_words)

['excuse', 'kindred', 'grease', 'laughter', 'day', 'genius', 'home', 'sensation', 'education', 'water', 'morning', 'pair', 'thanksgiving', 'allusion', 'engagement', 'failure', 'solder', 'pillow', 'totem', 'atmosphere', 'purpose', 'afternoon', 'tummy', 'everything', 'ear', 'step', 'bath', 'paste', 'touch', 'bedroom', 'rose', 'black', 'night', 'anything', 'ecstasy', 'manner', 'dark', 'wind', 'glass', 'just', 'trace', 'navel', 'ease', 'singer', 'king', 'body', 'mess', 'resemblance', 'being', 'adult', 'enchanter', 'mortality', 'age', 'ladder', 'advantage', 'attire', 'text', 'bough', 'desire', 'knee', 'perfume', 'verse', 'garment', 'bosom', 'doubt', 'vision', 'creature', 'beloved', 'birth', 'area', 'concrete', 'sun', 'world', 'mother', 'rest', 'speaking', 'charm', 'rock', 'piece', 'chin', 'wantonness', 'recalibration', 'bahram', 'goddess', 'joint', 'wonder', 'ferryboat', '_vocabilary_', 'hair', 'rubble', 'element', 'tone', 'stream', 'flutter', 'light', 'pleasure', 'apollo', 'breast', 'moral

In [16]:
#getting tagged grammar data
lines = open("./all-tags-2026-04-03.jsonl").readlines()
sentence_objs = [json.loads(item) for item in lines]

import random

tagged_sentences = [item for item in sentence_objs if len(item['label'])>0]
random.sample(tagged_sentences,5)

from collections import defaultdict
#part of speech
pos = defaultdict(list)

for item in tagged_sentences:
    for start, end, tag in item['label']:
        tagged_text = item['text'][start:end].strip()
        pos[tag].append(tagged_text)

pos.keys()

dict_keys(['prepositional phrase', 'noun (singular)', 'adjective', 'verb: transitive', 'verb: simple past', 'noun phrase', 'verb: intransitive', 'verb: uninflected', 'noun (plural)', 'verb: past participle', 'verb: present participle', 'adverb', 'verb: 3rd person singular present tense'])

In [10]:
pip install tracery

Note: you may need to restart the kernel to use updated packages.


In [11]:
#combining both data set 
pos_positive = defaultdict(list)
pos_negative = defaultdict(list)

#"adjective": ["dark", "warm", "little"...]                       
for tag, words in pos.items():
    for word in words:
        if word.lower() in positive_words:
            pos_positive[tag].append(word)
        if word.lower() in negative_words:
            pos_negative[tag].append(word)

In [12]:
# check my combined dataset
#pos_positive
print("5 random positive adjective: "+ ", ".join(random.sample(pos_positive["adjective"],5)))
print("5 random negative noun: "+ ", ".join(random.sample(pos_negative["noun (singular)"],5)))


5 random positive adjective: voluntary, ok, elder, Warm, near
5 random negative noun: modesty, Ball, drawing, violence, destruction


# Story structure made with tracery


In [14]:
import tracery
from tracery.modifiers import base_english

rules = {
    "origin": [
        "Once upon a time there was a #pos_adj# witch who lived in a #pos_noun#.\n "
        "#she_was#\n "
        "#she_thought#\n"
        "#the_witch#\n "
        "#she_said#\n "
        "#then_she#\n"
    ],

    "she_was": [
        "She was #pos_adj# and full of #pos_noun#.",
        "She was the most #pos_adj# creature in the #pos_noun#.",
    ],
    "she_thought": [
        "She thought of the #neg_noun# and felt #neg_adj#.",
        "She thought she could defeat the #neg_noun#.",
    ],
    "the_witch": [
        "The witch walked through the #neg_adj# #neg_noun#.",
        "The witch found a #neg_adj# #neg_noun# blocking the path.",
    ],
    "she_said": [
        "She said, I will not let the #neg_noun# win.",
        "She said, no #neg_noun# can stop a #pos_adj# heart.",
    ],
    "then_she": [
        "Then she defeated the #neg_noun# and all was #pos_adj# again, happily.",
        "Then she filled the #neg_noun# with #pos_noun# and lived happily ever after.",
    ],

    "pos_adj": pos_positive["adjective"],
    "neg_adj": pos_negative["adjective"],
    "pos_noun": pos_positive["noun (singular)"],
    "neg_noun": pos_negative["noun (singular)"],
    "pos_noun_pl": pos_positive["noun (plural)"],
    "neg_noun_pl": pos_negative["noun (plural)"],
    "verb_past": pos["verb: simple past"],
    "adverb": pos["adverb"],
}

grammar = tracery.Grammar(rules)
grammar.add_modifiers(base_english)

story = grammar.flatten("#origin#")
story = story.replace("witch", "🧚‍♀️ witch 🧚‍♀️")
for line in story.split("\n"):
    print(textwrap.fill(line.strip(), width=100))


Once upon a time there was a comic 🧚‍♀️ witch 🧚‍♀️ who lived in a love.
She was the most sentimental creature in the heat.
She thought she could defeat the Floor.
The 🧚‍♀️ witch 🧚‍♀️ found a noble saint blocking the path.
She said, no world can stop a squalid heart.
Then she filled the shower with anyone and lived happily ever after.



In [35]:
import tracery
from tracery.modifiers import base_english

rules = {
    "origin": [
        "Once upon a time there was a #pos_adj# witch who lived in a #pos_noun#.\n "
        "#she_was#\n "
        "#she_thought#\n"
        "#the_witch#\n "
        "#she_said#\n "
        "#then_she#\n"
    ],

    "she_was": [
        "She was #pos_adj# and full of #pos_noun#.",
        "She was the most #pos_adj# person in the #pos_noun#.",
    ],
    "she_thought": [
        "She thought of the #neg_noun# and felt #neg_adj#.",
        "She thought of the #pos_noun# and felt #pos_adj#.",
        "She thought she could defeat the #neg_noun#.",
        #"She thought the #neg_noun# would #verb_uninflected#",

    ],
    "the_witch": [
        "The witch walked through the #neg_adj# #neg_noun#.",
        "The witch found a #neg_adj# #neg_noun# blocking the path.",
        "The witch decided to #verb_past# with #pos_noun#",
    ],
    "she_said": [
        "She said, I will not let the #neg_noun# win.",
        "She said, no #neg_noun# can stop a #pos_adj# heart.",
    ],
    "then_she": [
        "Then she defeated the #neg_noun# and all was #pos_adj# again, happily.",
        "Then she filled the #neg_noun# with #pos_noun# and lived happily ever after.",
    ],

    "pos_adj": pos_positive["adjective"],
    "neg_adj": pos_negative["adjective"],
    "pos_noun": pos_positive["noun (singular)"],
    "neg_noun": pos_negative["noun (singular)"],
    "pos_noun_pl": pos_positive["noun (plural)"],
    "neg_noun_pl": pos_negative["noun (plural)"],
    "verb_past": pos["verb: simple past"],
    "verb_uninflected": pos["verb: uninflected"],
    "adverb": pos["adverb"],
}

grammar = tracery.Grammar(rules)
grammar.add_modifiers(base_english)

story = grammar.flatten("#origin#")
story = story.replace("witch", "🧚‍♀️ witch 🧚‍♀️")
for line in story.split("\n"):
    print(textwrap.fill(line.strip(), width=100))

Once upon a time there was a simple 🧚‍♀️ witch 🧚‍♀️ who lived in a text.
She was the most beautiful person in the conversation.
She thought of the cloth and felt white.
The 🧚‍♀️ witch 🧚‍♀️ walked through the only void.
She said, I will not let the anything win.
Then she filled the decrease with rest and lived happily ever after.

